# Structure B Item Query Decoder — V2 Accuracy Experiment

현재 `KoELECTRA-base + Item Query Transformer Decoder` 구조는 유지하고 학습 전략만 개선한 실험용 노트북입니다.

개선 항목:
- **Differential learning rate**: pretrained encoder와 새 decoder/head에 서로 다른 learning rate 적용
- **Multi-order oversampling**: 2-item / 3-item 주문을 학습 시 더 자주 샘플링
- **Class-weighted loss**: 불균형한 intent 및 item-active 라벨에 가중치 적용
- **Granular evaluation**: 전체뿐 아니라 1-item / 2-item / 3-item 주문 Exact Match를 별도로 측정
- **Error export**: 오답을 CSV로 저장하여 hard-example augmentation에 사용

운영 모델 구조 자체는 변경하지 않습니다.

## 1. 패키지 설치

In [ ]:
%pip -q install -U "transformers==4.46.3" "scikit-learn>=1.4" "pandas>=2.0" "tqdm>=4.66"

## 2. 환경 및 하이퍼파라미터

In [ ]:
import json
import math
import os
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 42

MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
MAX_LENGTH = 96
MAX_ITEMS = 3

BATCH_SIZE = 8
EPOCHS = 15
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
PATIENCE = 4
VALID_RATIO = 0.10

# 핵심 변경 1: pretrained encoder와 새 decoder/head의 LR 분리
ENCODER_LR = 1e-5
NEW_LAYER_LR = 5e-5

# 핵심 변경 2: multi-order 학습 비중 증가
ITEM_COUNT_SAMPLE_WEIGHT = {
    0: 1.0,   # 비-ORDER
    1: 1.0,
    2: 1.8,
    3: 2.5,
}

# 핵심 변경 3: class-weighted loss
USE_INTENT_CLASS_WEIGHT = True
USE_ACTIVE_CLASS_WEIGHT = True

GRAD_CLIP_NORM = 1.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    print("cuda:", torch.version.cuda)

## 3. 프로젝트 경로 및 데이터 불러오기

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (
            (candidate / "data" / "structure_b_train_valid.jsonl").exists()
            and (candidate / "data" / "structure_b_test.jsonl").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. "
        "yulllee0829/pumpkin 저장소 안에서 이 노트북을 실행하거나 "
        "data/structure_b_train_valid.jsonl, data/structure_b_test.jsonl 경로를 확인하세요."
    )

PROJECT_ROOT = find_project_root()
TRAIN_VALID_PATH = PROJECT_ROOT / "data" / "structure_b_train_valid.jsonl"
TEST_PATH = PROJECT_ROOT / "data" / "structure_b_test.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "nlu" / "nlu_model_results" / "item_query_v2_accuracy"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_jsonl(path: Path) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"{path.name}:{line_no} JSON 오류: {exc}") from exc
    return records

all_train_valid_records = load_jsonl(TRAIN_VALID_PATH)
test_records = load_jsonl(TEST_PATH)

print("project_root:", PROJECT_ROOT)
print("train_valid:", len(all_train_valid_records))
print("test:", len(test_records))
print("output_dir:", OUTPUT_DIR)

## 4. Train / Validation 분할

In [ ]:
def item_count(record: dict[str, Any]) -> int:
    if str(record.get("intent") or "") != "ORDER":
        return 0
    return min(len(record.get("items") or []), MAX_ITEMS)

def split_key(record: dict[str, Any]) -> str:
    intent = str(record.get("intent") or "UNKNOWN")
    status = str(record.get("order_status") or "NONE")
    count = item_count(record)
    # ORDER는 item 수와 status를 함께 보존하고, 비-ORDER는 intent 단위로 보존
    if intent == "ORDER":
        return f"{intent}|{status}|items={count}"
    return intent

def stratified_group_split(
    records: list[dict[str, Any]],
    valid_ratio: float = 0.1,
    seed: int = 42,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    rng = random.Random(seed)
    groups: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for record in records:
        groups[split_key(record)].append(record)

    train_records, valid_records = [], []
    for key, group in groups.items():
        group = list(group)
        rng.shuffle(group)

        if len(group) <= 1:
            train_records.extend(group)
            continue

        n_valid = max(1, int(round(len(group) * valid_ratio)))
        n_valid = min(n_valid, len(group) - 1)

        valid_records.extend(group[:n_valid])
        train_records.extend(group[n_valid:])

    rng.shuffle(train_records)
    rng.shuffle(valid_records)
    return train_records, valid_records

train_records, valid_records = stratified_group_split(
    all_train_valid_records,
    valid_ratio=VALID_RATIO,
    seed=SEED,
)

def summarize(records: list[dict[str, Any]], name: str) -> None:
    intents = Counter(str(r.get("intent") or "UNKNOWN") for r in records)
    item_counts = Counter(item_count(r) for r in records)
    print(f"\n[{name}] n={len(records)}")
    print("intent:", dict(intents))
    print("item_count:", dict(sorted(item_counts.items())))

summarize(train_records, "train")
summarize(valid_records, "valid")
summarize(test_records, "test")

## 5. 라벨 맵

In [ ]:
# 현재 운영 스키마와 동일한 intent label set 유지
# PAYMENT는 현재 canonical dataset에 positive sample이 없지만
# 출력 스키마 호환성을 위해 class는 유지합니다.
INTENT_LABELS = [
    "ORDER",
    "MODIFY",
    "CANCEL",
    "AFFIRM",
    "DENY",
    "GUIDE",
    "PAYMENT",
    "UNKNOWN",
]

STATUS_LABELS = [
    "NONE",
    "VALID",
    "INCOMPLETE",
    "CONFLICT",
    "OUT_OF_POLICY",
    "UNPARSABLE",
]

MENU_LABELS = [
    "NONE",
    "아메리카노",
    "카페라떼",
    "바닐라라떼",
    "레몬에이드",
    "딸기스무디",
]

TEMPERATURE_LABELS = ["NONE", "ICE", "HOT"]
QUANTITY_LABELS = ["NONE"] + [str(v) for v in range(1, 21)]

def make_maps(labels: list[str]):
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}
    return label2id, id2label

intent2id, id2intent = make_maps(INTENT_LABELS)
status2id, id2status = make_maps(STATUS_LABELS)
menu2id, id2menu = make_maps(MENU_LABELS)
temperature2id, id2temperature = make_maps(TEMPERATURE_LABELS)
quantity2id, id2quantity = make_maps(QUANTITY_LABELS)

LABEL_MAPS = {
    "intent": {"label2id": intent2id, "id2label": id2intent},
    "order_status": {"label2id": status2id, "id2label": id2status},
    "menu": {"label2id": menu2id, "id2label": id2menu},
    "temperature": {"label2id": temperature2id, "id2label": id2temperature},
    "quantity": {"label2id": quantity2id, "id2label": id2quantity},
}

print("intent labels:", INTENT_LABELS)

## 6. Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def normalize_status(record: dict[str, Any]) -> str:
    if str(record.get("intent") or "") != "ORDER":
        return "NONE"
    status = str(record.get("order_status") or "UNPARSABLE")
    return status if status in status2id else "UNPARSABLE"

def normalize_menu(value: Any) -> str:
    text = str(value).strip() if value is not None else "NONE"
    aliases = {
        "라떼": "카페라떼",
        "카페라테": "카페라떼",
        "바닐라 라떼": "바닐라라떼",
        "바닐라라테": "바닐라라떼",
    }
    text = aliases.get(text, text)
    return text if text in menu2id else "NONE"

def normalize_temperature(value: Any) -> str:
    text = str(value).strip().upper() if value is not None else "NONE"
    return text if text in temperature2id else "NONE"

def normalize_quantity(value: Any) -> str:
    if value is None:
        return "NONE"
    try:
        q = int(value)
    except (TypeError, ValueError):
        return "NONE"
    text = str(q)
    return text if text in quantity2id else "NONE"

class StructureBDataset(Dataset):
    def __init__(self, records: list[dict[str, Any]]):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index: int):
        record = self.records[index]
        text = str(record.get("text") or "").strip()
        intent = str(record.get("intent") or "UNKNOWN")

        encoded = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_tensors="pt",
        )

        item_active = [0] * MAX_ITEMS
        menu_labels = [menu2id["NONE"]] * MAX_ITEMS
        temperature_labels = [temperature2id["NONE"]] * MAX_ITEMS
        quantity_labels = [quantity2id["NONE"]] * MAX_ITEMS

        if intent == "ORDER":
            items = list(record.get("items") or [])[:MAX_ITEMS]
            for i, item in enumerate(items):
                item_active[i] = 1
                menu_labels[i] = menu2id[normalize_menu(item.get("menu"))]
                temperature_labels[i] = temperature2id[
                    normalize_temperature(item.get("temperature"))
                ]
                quantity_labels[i] = quantity2id[
                    normalize_quantity(item.get("quantity"))
                ]

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "intent_label": torch.tensor(intent2id[intent], dtype=torch.long),
            "status_label": torch.tensor(
                status2id[normalize_status(record)], dtype=torch.long
            ),
            "item_active": torch.tensor(item_active, dtype=torch.long),
            "menu_labels": torch.tensor(menu_labels, dtype=torch.long),
            "temperature_labels": torch.tensor(
                temperature_labels, dtype=torch.long
            ),
            "quantity_labels": torch.tensor(quantity_labels, dtype=torch.long),
            "sample_index": torch.tensor(index, dtype=torch.long),
        }

train_dataset = StructureBDataset(train_records)
valid_dataset = StructureBDataset(valid_records)
test_dataset = StructureBDataset(test_records)

print(len(train_dataset), len(valid_dataset), len(test_dataset))

## 7. Multi-order Oversampling

In [ ]:
train_sample_weights = torch.tensor(
    [
        ITEM_COUNT_SAMPLE_WEIGHT.get(item_count(record), 1.0)
        for record in train_records
    ],
    dtype=torch.double,
)

train_sampler = WeightedRandomSampler(
    weights=train_sample_weights,
    num_samples=len(train_sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=2,
    pin_memory=DEVICE.type == "cuda",
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=DEVICE.type == "cuda",
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=DEVICE.type == "cuda",
)

print("sample weights by item count:", ITEM_COUNT_SAMPLE_WEIGHT)

## 8. 모델 — 기존 구조 유지

In [ ]:
class ItemQueryDecoderModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.encoder.config.hidden_size

        self.intent_head = nn.Sequential(
            nn.Dropout(0.15),
            nn.Linear(hidden_size, len(INTENT_LABELS)),
        )
        self.status_head = nn.Sequential(
            nn.Dropout(0.15),
            nn.Linear(hidden_size, len(STATUS_LABELS)),
        )

        self.item_queries = nn.Parameter(
            torch.randn(MAX_ITEMS, hidden_size) * 0.02
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_size,
            nhead=4,
            dim_feedforward=hidden_size * 4,
            dropout=0.15,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.item_decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=2,
        )
        self.item_norm = nn.LayerNorm(hidden_size)

        self.active_head = nn.Linear(hidden_size, 2)
        self.menu_head = nn.Linear(hidden_size, len(MENU_LABELS))
        self.temperature_head = nn.Linear(
            hidden_size, len(TEMPERATURE_LABELS)
        )
        self.quantity_head = nn.Linear(
            hidden_size, len(QUANTITY_LABELS)
        )

    def forward(self, input_ids, attention_mask):
        memory = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state

        cls = memory[:, 0]

        batch_size = input_ids.size(0)
        queries = self.item_queries.unsqueeze(0).expand(
            batch_size, -1, -1
        )

        decoded = self.item_decoder(
            tgt=queries,
            memory=memory,
            memory_key_padding_mask=(attention_mask == 0),
        )
        decoded = self.item_norm(decoded)

        return {
            "intent_logits": self.intent_head(cls),
            "status_logits": self.status_head(cls),
            "active_logits": self.active_head(decoded),
            "menu_logits": self.menu_head(decoded),
            "temperature_logits": self.temperature_head(decoded),
            "quantity_logits": self.quantity_head(decoded),
        }

model = ItemQueryDecoderModel().to(DEVICE)
print("model ready")

## 9. Class Weight 및 Loss

In [ ]:
def inverse_sqrt_class_weights(
    label_ids: list[int],
    num_classes: int,
) -> torch.Tensor:
    counts = np.bincount(label_ids, minlength=num_classes).astype(np.float64)
    weights = np.zeros(num_classes, dtype=np.float64)

    nonzero = counts > 0
    weights[nonzero] = 1.0 / np.sqrt(counts[nonzero])
    if nonzero.any():
        weights[nonzero] /= weights[nonzero].mean()

    # 학습 샘플이 없는 클래스는 loss에 사용되지 않도록 0
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

intent_label_ids = [
    intent2id[str(r.get("intent") or "UNKNOWN")]
    for r in train_records
]
intent_class_weights = inverse_sqrt_class_weights(
    intent_label_ids,
    len(INTENT_LABELS),
)

# 각 query의 active/inactive를 모두 합쳐 분포 계산
active_targets = []
for record in train_records:
    count = item_count(record)
    active_targets.extend(
        [1 if i < count else 0 for i in range(MAX_ITEMS)]
    )

active_class_weights = inverse_sqrt_class_weights(
    active_targets,
    2,
)

intent_ce = nn.CrossEntropyLoss(
    weight=intent_class_weights if USE_INTENT_CLASS_WEIGHT else None
)
status_ce = nn.CrossEntropyLoss()
active_ce = nn.CrossEntropyLoss(
    weight=active_class_weights if USE_ACTIVE_CLASS_WEIGHT else None
)
slot_ce = nn.CrossEntropyLoss()

print("intent weights:")
for label, weight in zip(INTENT_LABELS, intent_class_weights.detach().cpu().tolist()):
    print(f"  {label:16s} {weight:.4f}")

print("active weights:", active_class_weights.detach().cpu().tolist())

def compute_loss(
    outputs: dict[str, torch.Tensor],
    batch: dict[str, torch.Tensor],
):
    intent_loss = intent_ce(
        outputs["intent_logits"],
        batch["intent_label"],
    )

    status_loss = status_ce(
        outputs["status_logits"],
        batch["status_label"],
    )

    active_loss = active_ce(
        outputs["active_logits"].reshape(-1, 2),
        batch["item_active"].reshape(-1),
    )

    active_mask = batch["item_active"].reshape(-1) == 1

    if active_mask.any():
        menu_loss = slot_ce(
            outputs["menu_logits"].reshape(-1, len(MENU_LABELS))[active_mask],
            batch["menu_labels"].reshape(-1)[active_mask],
        )
        temperature_loss = slot_ce(
            outputs["temperature_logits"]
            .reshape(-1, len(TEMPERATURE_LABELS))[active_mask],
            batch["temperature_labels"].reshape(-1)[active_mask],
        )
        quantity_loss = slot_ce(
            outputs["quantity_logits"]
            .reshape(-1, len(QUANTITY_LABELS))[active_mask],
            batch["quantity_labels"].reshape(-1)[active_mask],
        )
    else:
        zero = outputs["intent_logits"].sum() * 0.0
        menu_loss = zero
        temperature_loss = zero
        quantity_loss = zero

    total_loss = (
        1.0 * intent_loss
        + 1.0 * status_loss
        + 1.0 * active_loss
        + 1.3 * menu_loss
        + 1.0 * temperature_loss
        + 1.0 * quantity_loss
    )

    return total_loss, {
        "intent": intent_loss.detach().item(),
        "status": status_loss.detach().item(),
        "active": active_loss.detach().item(),
        "menu": menu_loss.detach().item(),
        "temperature": temperature_loss.detach().item(),
        "quantity": quantity_loss.detach().item(),
    }

## 10. Differential Learning Rate Optimizer

In [ ]:
encoder_params = list(model.encoder.parameters())

new_layer_modules = [
    model.intent_head,
    model.status_head,
    model.item_decoder,
    model.item_norm,
    model.active_head,
    model.menu_head,
    model.temperature_head,
    model.quantity_head,
]

new_layer_params = [model.item_queries]
for module in new_layer_modules:
    new_layer_params.extend(list(module.parameters()))

optimizer = torch.optim.AdamW(
    [
        {
            "params": encoder_params,
            "lr": ENCODER_LR,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": new_layer_params,
            "lr": NEW_LAYER_LR,
            "weight_decay": WEIGHT_DECAY,
        },
    ]
)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

print("encoder lr:", ENCODER_LR)
print("new layer lr:", NEW_LAYER_LR)
print("total steps:", total_steps)

## 11. 평가 함수

In [ ]:
def batch_to_device(batch: dict[str, torch.Tensor]):
    return {
        k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v)
        for k, v in batch.items()
    }

def masked_accuracy(true_values, pred_values) -> float:
    if len(true_values) == 0:
        return float("nan")
    return float(np.mean(np.asarray(true_values) == np.asarray(pred_values)))

@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    source_records: list[dict[str, Any]],
    desc: str = "eval",
):
    model.eval()

    total_loss = 0.0
    n_batches = 0

    true_intents, pred_intents = [], []
    true_statuses, pred_statuses = [], []

    true_active, pred_active = [], []
    true_menu, pred_menu = [], []
    true_temp, pred_temp = [], []
    true_qty, pred_qty = [], []

    rows = []

    for batch in tqdm(loader, desc=desc, leave=False):
        indices = batch["sample_index"].tolist()
        batch = batch_to_device(batch)

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )

        loss, _ = compute_loss(outputs, batch)
        total_loss += float(loss.item())
        n_batches += 1

        intent_pred = outputs["intent_logits"].argmax(dim=-1)
        status_pred = outputs["status_logits"].argmax(dim=-1)
        active_pred = outputs["active_logits"].argmax(dim=-1)
        menu_pred = outputs["menu_logits"].argmax(dim=-1)
        temp_pred = outputs["temperature_logits"].argmax(dim=-1)
        qty_pred = outputs["quantity_logits"].argmax(dim=-1)

        true_intents.extend(batch["intent_label"].cpu().tolist())
        pred_intents.extend(intent_pred.cpu().tolist())

        true_statuses.extend(batch["status_label"].cpu().tolist())
        pred_statuses.extend(status_pred.cpu().tolist())

        true_active.extend(batch["item_active"].reshape(-1).cpu().tolist())
        pred_active.extend(active_pred.reshape(-1).cpu().tolist())

        for b in range(batch["input_ids"].size(0)):
            record_index = int(indices[b])
            record = source_records[record_index]
            gold_count = item_count(record)

            gold_intent = int(batch["intent_label"][b].item())
            pred_intent = int(intent_pred[b].item())
            gold_status = int(batch["status_label"][b].item())
            pred_status = int(status_pred[b].item())

            # 활성 gold item의 slot accuracy
            for q in range(MAX_ITEMS):
                if int(batch["item_active"][b, q].item()) == 1:
                    true_menu.append(int(batch["menu_labels"][b, q].item()))
                    pred_menu.append(int(menu_pred[b, q].item()))

                    true_temp.append(
                        int(batch["temperature_labels"][b, q].item())
                    )
                    pred_temp.append(int(temp_pred[b, q].item()))

                    true_qty.append(
                        int(batch["quantity_labels"][b, q].item())
                    )
                    pred_qty.append(int(qty_pred[b, q].item()))

            intent_ok = gold_intent == pred_intent
            status_ok = gold_status == pred_status

            active_ok = bool(torch.equal(
                batch["item_active"][b].detach().cpu(),
                active_pred[b].detach().cpu(),
            ))

            item_frame_ok = True
            if gold_count > 0:
                for q in range(MAX_ITEMS):
                    gold_is_active = int(batch["item_active"][b, q].item()) == 1
                    pred_is_active = int(active_pred[b, q].item()) == 1

                    if gold_is_active != pred_is_active:
                        item_frame_ok = False
                        break

                    if gold_is_active:
                        if (
                            int(batch["menu_labels"][b, q].item())
                            != int(menu_pred[b, q].item())
                            or int(batch["temperature_labels"][b, q].item())
                            != int(temp_pred[b, q].item())
                            or int(batch["quantity_labels"][b, q].item())
                            != int(qty_pred[b, q].item())
                        ):
                            item_frame_ok = False
                            break

            frame_ok = intent_ok and status_ok and active_ok and item_frame_ok

            rows.append({
                "record_index": record_index,
                "text": str(record.get("text") or ""),
                "intent": str(record.get("intent") or "UNKNOWN"),
                "item_count": gold_count,
                "order_status": normalize_status(record),
                "intent_correct": intent_ok,
                "status_correct": status_ok,
                "active_correct": active_ok,
                "item_frame_correct": item_frame_ok,
                "frame_correct": frame_ok,
            })

    result_df = pd.DataFrame(rows)

    metrics = {
        "loss": total_loss / max(n_batches, 1),
        "intent_accuracy": accuracy_score(true_intents, pred_intents),
        "intent_macro_f1": f1_score(
            true_intents,
            pred_intents,
            average="macro",
            zero_division=0,
        ),
        "status_accuracy": accuracy_score(true_statuses, pred_statuses),
        "status_macro_f1": f1_score(
            true_statuses,
            pred_statuses,
            average="macro",
            zero_division=0,
        ),
        "item_active_accuracy": accuracy_score(true_active, pred_active),
        "menu_accuracy": masked_accuracy(true_menu, pred_menu),
        "temperature_accuracy": masked_accuracy(true_temp, pred_temp),
        "quantity_accuracy": masked_accuracy(true_qty, pred_qty),
        "frame_accuracy": float(result_df["frame_correct"].mean()),
    }

    order_df = result_df[result_df["intent"] == "ORDER"]
    if len(order_df):
        metrics["order_exact_match"] = float(
            order_df["frame_correct"].mean()
        )
    else:
        metrics["order_exact_match"] = float("nan")

    for count in (1, 2, 3):
        subset = order_df[order_df["item_count"] == count]
        metrics[f"order_{count}item_exact_match"] = (
            float(subset["frame_correct"].mean())
            if len(subset)
            else float("nan")
        )
        metrics[f"order_{count}item_n"] = int(len(subset))

    return metrics, result_df

## 12. 학습

In [ ]:
best_score = -1.0
best_epoch = 0
epochs_without_improvement = 0
history = []

best_model_path = OUTPUT_DIR / "best_model.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()

    running_loss = 0.0
    batch_count = 0

    progress = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}")

    for batch in progress:
        batch = batch_to_device(batch)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )
            loss, loss_parts = compute_loss(outputs, batch)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP_NORM,
        )

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += float(loss.item())
        batch_count += 1

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            enc_lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            new_lr=f"{optimizer.param_groups[1]['lr']:.2e}",
        )

    train_loss = running_loss / max(batch_count, 1)

    valid_metrics, _ = evaluate(
        model,
        valid_loader,
        valid_records,
        desc="valid",
    )

    # 복수 주문 성능을 더 중요하게 보되 single도 함께 고려
    valid_single = valid_metrics.get("order_1item_exact_match", float("nan"))
    valid_2 = valid_metrics.get("order_2item_exact_match", float("nan"))
    valid_3 = valid_metrics.get("order_3item_exact_match", float("nan"))

    available = [
        (0.35, valid_single),
        (0.35, valid_2),
        (0.30, valid_3),
    ]
    weighted = [(w, v) for w, v in available if not math.isnan(v)]
    checkpoint_score = (
        sum(w * v for w, v in weighted) / sum(w for w, _ in weighted)
        if weighted
        else valid_metrics["frame_accuracy"]
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "checkpoint_score": checkpoint_score,
        **{f"valid_{k}": v for k, v in valid_metrics.items()},
    }
    history.append(row)

    print(
        f"epoch={epoch} "
        f"train_loss={train_loss:.4f} "
        f"valid_frame={valid_metrics['frame_accuracy']:.4f} "
        f"single={valid_single:.4f} "
        f"2item={valid_2:.4f} "
        f"3item={valid_3:.4f} "
        f"score={checkpoint_score:.4f}"
    )

    if checkpoint_score > best_score:
        best_score = checkpoint_score
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "max_items": MAX_ITEMS,
            "label_maps": LABEL_MAPS,
            "training_config": {
                "encoder_lr": ENCODER_LR,
                "new_layer_lr": NEW_LAYER_LR,
                "batch_size": BATCH_SIZE,
                "epochs": EPOCHS,
                "weight_decay": WEIGHT_DECAY,
                "warmup_ratio": WARMUP_RATIO,
                "item_count_sample_weight": ITEM_COUNT_SAMPLE_WEIGHT,
                "use_intent_class_weight": USE_INTENT_CLASS_WEIGHT,
                "use_active_class_weight": USE_ACTIVE_CLASS_WEIGHT,
            },
            "best_epoch": best_epoch,
            "best_checkpoint_score": best_score,
        }, best_model_path)

        tokenizer.save_pretrained(OUTPUT_DIR / "tokenizer")
        model.encoder.config.save_pretrained(OUTPUT_DIR / "encoder_config")
        print("saved:", best_model_path)
    else:
        epochs_without_improvement += 1

    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "training_history.csv",
        index=False,
        encoding="utf-8-sig",
    )

    if epochs_without_improvement >= PATIENCE:
        print("early stopping")
        break

print("best_epoch:", best_epoch)
print("best_score:", best_score)

## 13. Best Model 로드 및 최종 Test

In [ ]:
checkpoint = torch.load(
    best_model_path,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

test_metrics, test_predictions = evaluate(
    model,
    test_loader,
    test_records,
    desc="test",
)

print(json.dumps(test_metrics, ensure_ascii=False, indent=2))

with (OUTPUT_DIR / "test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

test_predictions.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

errors = test_predictions[
    ~test_predictions["frame_correct"]
].copy()

errors.to_csv(
    OUTPUT_DIR / "test_errors.csv",
    index=False,
    encoding="utf-8-sig",
)

print("errors:", len(errors))
print("saved:", OUTPUT_DIR)

## 14. 논문용 결과표

In [ ]:
paper_table = pd.DataFrame([{
    "Model": "KoELECTRA + Item Query Decoder V2",
    "Intent Macro-F1": test_metrics["intent_macro_f1"],
    "Menu Acc.": test_metrics["menu_accuracy"],
    "Temp. Acc.": test_metrics["temperature_accuracy"],
    "Qty. Acc.": test_metrics["quantity_accuracy"],
    "Single EM": test_metrics["order_1item_exact_match"],
    "2-item EM": test_metrics["order_2item_exact_match"],
    "3-item EM": test_metrics["order_3item_exact_match"],
    "Overall Order EM": test_metrics["order_exact_match"],
    "Frame Acc.": test_metrics["frame_accuracy"],
}])

display(paper_table.round(4))

paper_table.to_csv(
    OUTPUT_DIR / "paper_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

## 15. 오답 유형 확인

In [ ]:
error_summary = (
    errors
    .groupby(["intent", "item_count"], dropna=False)
    .size()
    .reset_index(name="errors")
    .sort_values("errors", ascending=False)
)

display(error_summary)
display(
    errors[
        [
            "text",
            "intent",
            "item_count",
            "order_status",
            "intent_correct",
            "status_correct",
            "active_correct",
            "item_frame_correct",
        ]
    ].head(100)
)

## 실행 결과물

학습이 끝나면 아래 경로에 저장됩니다.

```text
experiments/nlu/nlu_model_results/item_query_v2_accuracy/
├── best_model.pt
├── tokenizer/
├── encoder_config/
├── training_history.csv
├── test_metrics.json
├── test_predictions.csv
├── test_errors.csv
└── paper_metrics.csv
```

`paper_metrics.csv`는 기존 모델과의 비교표 작성에 바로 사용할 수 있습니다.